In [5]:
from typing import TypedDict
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from langgraph.constants import START, END
from langgraph.graph import StateGraph

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

# 1. 定义状态
class OverAllState(TypedDict):
    topic: str
    poem: str
    joke: str

# 2. 定义节点
def node_a(state: OverAllState) -> OverAllState:
    response = model.invoke([HumanMessage(content=f"写一首关于{state['topic']}的唐诗")])
    return {"poem": response.content}

def node_b(state: OverAllState) -> OverAllState:
    response = model.invoke([HumanMessage(content=f"写一个关于{state['topic']}的笑话")])
    return {"joke": response.content}

# 3. 构建图
builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)

builder.add_edge(START, "node_a")
builder.add_edge(START, "node_b")
builder.add_edge("node_a", END)
builder.add_edge("node_b", END)

graph = builder.compile()
graph.invoke({"topic": "冯浚桀"})

{'topic': '冯浚桀',
 'poem': '《寄浚桀子》\n冯生才调世无伦，腹有诗书气自真。\n浚水清波澄玉宇，桀骜风骨立红尘。\n笔走龙蛇惊风雨，墨染云霞泣鬼神。\n他日若遂凌云志，莫忘初心与故人。\n\n注：本诗以“冯浚桀”三字为藏头，首联赞其才情卓绝，颔联以“浚水”喻其澄澈品性，“桀骜风骨”显其不羁气节。颈联化用“笔落惊风雨，诗成泣鬼神”之典，展现其文采斐然。尾联寄寓期许，勉其保持赤子之心，与旧友共勉。全诗对仗工整，气韵雄浑，既有唐诗豪迈之风，又含殷切勉励之情。',
 'joke': '关于“冯浚桀”的笑话，如果是朋友间的调侃（比如谐音或名字梗），下面这个段子也许能用：\n\n---\n\n**《关于冯浚桀的“严谨”》**  \n冯浚桀去应聘程序员，面试官问：“你觉得自己最大的优点是什么？”  \n他自信回答：“严谨，特别注重细节。”  \n面试官指了指电脑：“那你把这份代码跑一下，看看结果。”  \n冯浚桀盯着屏幕看了五分钟，严肃地说：“报告，这代码有个致命漏洞——它……它没写我的名字。”  \n\n---\n\n**《冯浚桀的冷笑话》**  \n朋友问冯浚桀：“你觉得自己像什么动物？”  \n他想了想：“像老虎，因为我的名字里有个‘桀’，听起来就很霸气。”  \n朋友又问：“那为什么你刚才被一只猫吓得跳上了桌子？”  \n他淡定回答：“因为那只猫的项圈上写着‘凌’——它是我命里的克星。”  \n\n---\n\n如果需要更具体的场景（比如校园、职场或情侣梗），可以告诉我，我再调整～ （笑）'}